# Making the virtual patients realistic: calibration to the real trial, with held-out checks

The dosing simulation draws patients from a published population model (Bai et al. 2024). Used as published, it produced far more days off target than the real trial,
so the synthetic data was not realistic. Here the synthetic patients are calibrated to real per-patient data, without overfitting:

- **Real reference:** the CURATE.AI trial's linear-rule arm (Khong 2025, source data), restricted to the **22 patients with an 8-10 ng/mL target** (the other 5 had a different, unreported range).
  The trial arm used exactly the rule simulated here, so the comparison is like for like. Hospital stays are resampled from these 22 real patients.
- **Few knobs:** residual noise of the trough, day-to-day clearance variation, start dose (days 1-3 were set by clinicians and are not reported), and two standard readings of the published variability (log-SD or CV).
- **Fit on 3 statistics, judge on 3 others never used for fitting.** If the held-out statistics are off, the synthetic data is not realistic in that respect, and that is reported rather than tuned away.

Code: `dosing/calib.py`.

In [1]:
import itertools
import numpy as np
import pandas as pd
import calib
from calib import simulate_linear

p = pd.read_csv("../data/khong2025_tacrolimus_rct_source_data.csv")
p = p[(p.Group == "PPM") & (p["Tac Range 8-10"] == "Y")]
REAL = dict(far=p["%Days >2"].mean(), mean_ttl=p["Mean TTL"].mean(), add_below=p.ADDBelow.mean(),
            add_above=p.ADDAbove.mean(), far_sd=p["%Days >2"].std(), mean_ttl_sd=p["Mean TTL"].std())
FIT = ["far", "mean_ttl", "add_below"]; HELD = ["add_above", "far_sd", "mean_ttl_sd"]
SE = {k: p[c].std() / np.sqrt(len(p)) for k, c in (("far", "%Days >2"), ("mean_ttl", "Mean TTL"), ("add_below", "ADDBelow"))}
STAYS = p.LOHosp.clip(upper=30).to_numpy()
cv2sd = lambda cv: float(np.sqrt(np.log(1 + cv**2)))
READINGS = {"log-SD": (0.666, 1.26, 2.36), "CV": (cv2sd(0.666), cv2sd(1.26), cv2sd(2.36))}
print("real targets (n=%d):" % len(p), {k: round(v, 3) for k, v in REAL.items()})

real targets (n=22): {'far': np.float64(0.247), 'mean_ttl': np.float64(8.684), 'add_below': np.float64(0.776), 'add_above': np.float64(0.576), 'far_sd': np.float64(0.187), 'mean_ttl_sd': np.float64(1.198)}


In [2]:
rows = []
for reading, (oc, ov, ok) in READINGS.items():
    calib.OM_CL, calib.OM_V, calib.OM_KA = oc, ov, ok
    for start, prop, iov in itertools.product([2, 3, 4, 5], [0.05, 0.10, 0.15, 0.20, 0.25], [0.0, 0.1, 0.2, 0.3]):
        calib.START = start
        s = simulate_linear(prop, iov, 300, STAYS, seed=1)
        rows.append(dict(reading=reading, start=start, prop=prop, iov=iov,
                         fit_dist=sum(((s[k] - REAL[k]) / SE[k]) ** 2 for k in FIT), **s))
GRID = pd.DataFrame(rows).sort_values("fit_dist")
print(GRID.groupby("reading").head(3).round(3).to_string(index=False))

reading  start  prop  iov  fit_dist   far  mean_ttl  far_sd  mean_ttl_sd  add_below  add_above
 log-SD      4  0.05  0.0     0.505 0.261     8.671   0.339        2.539      0.845      0.536
 log-SD      5  0.05  0.0     0.663 0.265     8.856   0.338        2.653      0.787      0.644
 log-SD      4  0.05  0.1     0.752 0.266     8.713   0.330        2.551      0.856      0.572
     CV      2  0.10  0.0     4.015 0.241     9.043   0.269        1.828      0.617      0.631
     CV      2  0.10  0.1     4.403 0.252     9.087   0.262        1.831      0.621      0.663
     CV      2  0.15  0.0     4.718 0.291     9.114   0.259        1.921      0.683      0.793


## Chosen setting and how real it looks

The log-SD reading fits the averages almost perfectly but makes patients about twice as varied as the real ones (held-out spread of mean trough about 2.5 vs 1.2), so it is rejected.
The CV reading is chosen: it fits the averages less tightly but is closer on the held-out spreads. Its best grid point: start 2 mg, 10% residual noise, no day-to-day clearance variation.

In [3]:
CHOSEN = dict(reading="CV", start=2, prop=0.10, iov=0.0)
calib.OM_CL, calib.OM_V, calib.OM_KA = READINGS["CV"]; calib.START = CHOSEN["start"]
SIM = simulate_linear(CHOSEN["prop"], CHOSEN["iov"], 2000, STAYS, seed=7)   # fresh seed, larger cohort
CHECK = pd.DataFrame({"real": REAL, "synthetic (n=2000)": SIM}).loc[FIT + HELD]
CHECK["used for fitting"] = ["yes"] * len(FIT) + ["no"] * len(HELD)
CHECK["synthetic / real"] = CHECK["synthetic (n=2000)"] / CHECK["real"]
print(CHECK.round(3).to_string())

              real  synthetic (n=2000) used for fitting  synthetic / real
far          0.247               0.286              yes             1.157
mean_ttl     8.684               8.881              yes             1.023
add_below    0.776               0.752              yes             0.970
add_above    0.576               0.633               no             1.098
far_sd       0.187               0.286               no             1.527
mean_ttl_sd  1.198               1.950               no             1.628


## What this says

- **Averages are realistic.** On a fresh synthetic cohort of 2,000, days far off target, mean trough, below-range deviation and the held-out above-range deviation are within about 2-16% of the real trial.
- **Patient-to-patient spread is not.** Synthetic patients are about 50-60% more varied than the real ones (held-out spreads). The published model comes from a different population (a single centre in China), and no small honest change fixed this.
- **Rejected explanations, kept for the record:** reading the variability as log-SDs (spread twice too large); clinicians titrating from day 2 (tested by starting the rule on day 2: troughs overshoot, worse on every statistic).
- Consequence for the dosing comparison: rules are compared on patients that are realistic on average but more varied than in the trial. How this changes the size of the differences between rules was not tested.